# Part 5E — Multimodal RAG (Notebook 09)

This notebook implements multimodal retrieval with:

1. OCR using `ollama run glm-ocr`
2. Vision understanding using `qwen3.5:4b`

Input source: first page of real ArXiv PDFs.


## Tutorial Goals

This notebook is a standalone, zero-to-hero tutorial with:

1. Concept explanation from first principles
2. Architecture and workflow breakdown
3. End-to-end implementation code
4. Real execution outputs and benchmark metrics
5. Practical analysis and production takeaways


## What is this technique?

        ### Definition and core concepts
        Multimodal RAG enriches retrieval and generation with non-text inputs such as document images.

        ### Why was this technique developed?
        Text-only pipelines miss visual/layout context and OCR-only documents.

        ### What limitations of traditional RAG does it solve?
        It addresses visual information loss and improves robustness on PDF/image-heavy corpora.

        ### Architecture and workflow diagram explanation

```mermaid
graph TD
    P[ArXiv PDF] --> I[First-page image]
    I --> O[glm-ocr text]
    I --> V[qwen3.5 vision summary]
    O --> F[Fusion retrieval]
    V --> F
    F --> G[Generation]
```


        ### Component-by-component breakdown
        PDF fetch, first-page rendering, OCR extraction, vision extraction, multimodal fusion, generation/evaluation.

        ### When should it be used in real-world systems?
        Use for scanned documents, figure/table-heavy docs, and any corpus where visual signal matters.

        ### Advantages and disadvantages
        **Advantages**
        - Captures visual evidence
- Improves retrieval breadth
- Better robustness to text extraction gaps

        **Disadvantages**
        - More ingestion complexity
- OCR noise
- Extra latency

        ### Comparison against standard RAG and other implemented RAG variants
        Compared with standard RAG, multimodal RAG has broader evidence access. Compared with CRAG/Agentic, it focuses on modality expansion.

        ### Implementation details and design decisions used in this project
        This project uses first-page ArXiv PDF images, CLI OCR via `ollama run glm-ocr`, and qwen3.5 vision summaries, then fuses both signals.


In [1]:
from __future__ import annotations

import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path('.').resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.rag_v2.data import load_base_corpus, load_papers_from_chunks
from src.rag_v2.retrieval import DenseRetriever, BM25Retriever, HybridRetriever
from src.rag_v2.metrics import build_keyword_eval_set, compute_retrieval_metrics, save_json


from src.rag_v2.multimodal import download_arxiv_pdf, pdf_first_page_to_png, run_glm_ocr_cli, run_qwen_vision
from src.rag_v2.agentic import llm_faithfulness
ART = PROJECT_ROOT / 'artifacts' / 'rag_v2'
ART.mkdir(parents=True, exist_ok=True)

index, chunks = load_base_corpus()
papers = load_papers_from_chunks(chunks)

display(Markdown(f"Loaded **{len(chunks):,} chunks** from **{len(papers):,} papers** (FAISS dim={index.d})."))


Loaded **30,084 chunks** from **4,000 papers** (FAISS dim=1024).

In [2]:
MM_DIR = ART / 'multimodal'
PDF_DIR = MM_DIR / 'pdfs'
IMG_DIR = MM_DIR / 'images'
for p in [MM_DIR, PDF_DIR, IMG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

arxiv_ids = ['1706.03762', '2005.14165']
records = []
download_errors = []
for aid in arxiv_ids:
    try:
        pdf_path = download_arxiv_pdf(aid, PDF_DIR / f'{aid}.pdf')
        image_path = pdf_first_page_to_png(pdf_path, IMG_DIR / f'{aid}_p1.png')
        records.append({'arxiv_id': aid, 'pdf_path': str(pdf_path), 'image_path': str(image_path)})
    except Exception as exc:
        download_errors.append({'arxiv_id': aid, 'error': str(exc)})

records_df = pd.DataFrame(records)
if not records_df.empty:
    display(records_df)
if download_errors:
    display(pd.DataFrame(download_errors))

if records_df.empty:
    raise RuntimeError('No multimodal assets available. Check PDF download/render prerequisites.')


,arxiv_id,pdf_path,image_path
0,1706.03762,/home/ahmad/AI/Github/agentic-rag-arxiv-resear...,/home/ahmad/AI/Github/agentic-rag-arxiv-resear...
1,2005.14165,/home/ahmad/AI/Github/agentic-rag-arxiv-resear...,/home/ahmad/AI/Github/agentic-rag-arxiv-resear...


In [3]:
ocr_rows = []
vision_rows = []
extract_errors = []

for r in records:
    img = Path(r['image_path'])

    try:
        ocr_text = run_glm_ocr_cli(img)
    except Exception as exc:
        ocr_text = ''
        extract_errors.append({'arxiv_id': r['arxiv_id'], 'stage': 'glm_ocr', 'error': str(exc)})

    try:
        vision_text = run_qwen_vision(
            img,
            question='Summarize this page and list key terms.',
            fallback_text=ocr_text,
        )
        if not vision_text.strip():
            extract_errors.append({'arxiv_id': r['arxiv_id'], 'stage': 'qwen_vision', 'error': 'empty_response'})
    except Exception as exc:
        vision_text = ''
        extract_errors.append({'arxiv_id': r['arxiv_id'], 'stage': 'qwen_vision', 'error': str(exc)})

    ocr_rows.append({**r, 'ocr_text': ocr_text})
    vision_rows.append({**r, 'vision_text': vision_text})

ocr_df = pd.DataFrame(ocr_rows)
vision_df = pd.DataFrame(vision_rows)

stats_df = pd.DataFrame(
    {
        'arxiv_id': [r['arxiv_id'] for r in records],
        'ocr_chars': ocr_df['ocr_text'].str.len(),
        'vision_chars': vision_df['vision_text'].str.len(),
    }
)

if extract_errors:
    display(pd.DataFrame(extract_errors))

stats_df


,arxiv_id,stage,error
0,1706.03762,qwen_vision,empty_response
1,2005.14165,qwen_vision,empty_response


,arxiv_id,ocr_chars,vision_chars
0,1706.03762,2982,0
1,2005.14165,2570,0


In [4]:
import ollama

mm_docs = []
for i, aid in enumerate(arxiv_ids):
    ocr_text = str(ocr_df.loc[i, 'ocr_text']) if i < len(ocr_df) else ''
    vision_text = str(vision_df.loc[i, 'vision_text']) if i < len(vision_df) else ''
    if ocr_text.strip():
        mm_docs.append({'doc_id': f'{aid}_ocr', 'paper_id': aid, 'source': 'ocr', 'text': ocr_text})
    if vision_text.strip():
        mm_docs.append({'doc_id': f'{aid}_vision', 'paper_id': aid, 'source': 'vision', 'text': vision_text})


def mm_retrieve(question: str, top_k: int = 4):
    q = question.lower()
    scored = []
    for d in mm_docs:
        s = sum(1 for tok in q.split() if tok in d['text'].lower())
        scored.append((s, d))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [x[1] for x in scored[:top_k]]


eval_questions = [
    'What is the main contribution of the Transformer paper?',
    'What does the GPT-3 paper emphasize about model scale?',
]

rows = []
for i, q in enumerate(eval_questions):
    docs = mm_retrieve(q, top_k=4)
    if not docs:
        rows.append(
            {
                'question': q,
                'sources': '',
                'faithfulness': np.nan,
                'latency_ms': np.nan,
                'reason': 'no_retrieved_docs',
                'llm_evaluated': False,
            }
        )
        continue

    ctx = "\n\n".join(f"[{d['source']}] {d['text'][:900]}" for d in docs)
    prompt = f"Use only this context.\n\nContext:\n{ctx}\n\nQuestion: {q}\nAnswer:"
    t0 = time.perf_counter()
    resp = ollama.chat(
        model='qwen3.5:4b',
        messages=[{'role': 'user', 'content': prompt}],
        options={'temperature': 0.1, 'num_gpu': 0, 'num_ctx': 2048, 'num_predict': 180},
    )
    latency = (time.perf_counter() - t0) * 1000
    answer = resp['message']['content'].strip()

    if i == 0:
        faith, reason = llm_faithfulness(q, answer, [d['text'] for d in docs], judge_model='granite4.1:8b')
        llm_eval = True
    else:
        faith, reason = np.nan, 'not_evaluated_in_light_mode'
        llm_eval = False

    rows.append(
        {
            'question': q,
            'sources': ','.join(sorted(set(d['source'] for d in docs))),
            'faithfulness': round(float(faith), 3) if faith == faith else np.nan,
            'latency_ms': round(float(latency), 2),
            'reason': reason,
            'llm_evaluated': llm_eval,
        }
    )

mm_eval_df = pd.DataFrame(rows)
mm_eval_df


,question,sources,faithfulness,latency_ms,reason,llm_evaluated
0,What is the main contribution of the Transform...,ocr,1.0,23999.79,The context explicitly states that the paper '...,True
1,What does the GPT-3 paper emphasize about mode...,ocr,NaN,20953.47,not_evaluated_in_light_mode,False


In [5]:
faith_rows = mm_eval_df['faithfulness'].dropna()
lat_rows = mm_eval_df['latency_ms'].dropna()

mm_summary = {
    'n_pdfs': len(arxiv_ids),
    'n_docs': len(mm_docs),
    'ocr_avg_chars': round(float(ocr_df['ocr_text'].str.len().mean()), 2),
    'vision_avg_chars': round(float(vision_df['vision_text'].str.len().mean()), 2),
    'faithfulness_mean': round(float(faith_rows.mean()), 4) if not faith_rows.empty else None,
    'llm_evaluated_rows': int(mm_eval_df['llm_evaluated'].sum()),
    'latency_p50_ms': round(float(lat_rows.quantile(0.50)), 2) if not lat_rows.empty else None,
    'latency_p95_ms': round(float(lat_rows.quantile(0.95)), 2) if not lat_rows.empty else None,
}


def clean_nan(v):
    return None if isinstance(v, float) and np.isnan(v) else v


eval_records = [{k: clean_nan(v) for k, v in row.items()} for row in mm_eval_df.to_dict(orient='records')]

out_json = ART / 'multimodal' / '09_multimodal_metrics.json'
out_json.parent.mkdir(parents=True, exist_ok=True)
save_json(out_json, {'summary': mm_summary, 'records': records, 'eval': eval_records})

mm_summary


{'n_pdfs': 2,
 'n_docs': 2,
 'ocr_avg_chars': 2776.0,
 'vision_avg_chars': 0.0,
 'faithfulness_mean': 1.0,
 'llm_evaluated_rows': 1,
 'latency_p50_ms': 22476.63,
 'latency_p95_ms': 23847.47}

In [6]:
if mm_summary['vision_avg_chars'] == 0.0:
    vision_note = 'qwen3.5:4b returned empty vision outputs in this environment; OCR carried retrieval and answer grounding in this run.'
else:
    vision_note = 'Both OCR and qwen vision branches contributed non-empty multimodal context.'

analysis = (
    "## Post-run Analysis (Real Results)\n\n"
    f"- PDFs processed: **{mm_summary['n_pdfs']}**\n"
    f"- OCR average length: **{mm_summary['ocr_avg_chars']} chars**\n"
    f"- Vision average length: **{mm_summary['vision_avg_chars']} chars**\n"
    f"- LLM-evaluated rows: **{mm_summary['llm_evaluated_rows']}**\n"
    f"- Mean faithfulness (evaluated rows only): **{mm_summary['faithfulness_mean']}**\n"
    f"- P50 latency: **{mm_summary['latency_p50_ms']} ms**\n"
    f"- P95 latency: **{mm_summary['latency_p95_ms']} ms**\n\n"
    f"- Observation: {vision_note}"
)
display(Markdown(analysis))


## Post-run Analysis (Real Results)

- PDFs processed: **2**
- OCR average length: **2776.0 chars**
- Vision average length: **0.0 chars**
- LLM-evaluated rows: **1**
- Mean faithfulness (evaluated rows only): **1.0**
- P50 latency: **22476.63 ms**
- P95 latency: **23847.47 ms**

- Observation: qwen3.5:4b returned empty vision outputs in this environment; OCR carried retrieval and answer grounding in this run.